In [1]:
!pip install huggingface_hub timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 

In [3]:
import os
import shutil
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from pathlib import Path
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm
import torch.nn.functional as F

# ================= CONFIGURATION =================
user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
REPO_ID = user_secrets.get_secret("REPO_ID")

BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/embeddings")
os.makedirs(OUTPUT_DIR, exist_ok=True)
# =================================================

IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class DroneEmbeddingDataset(Dataset):
    def __init__(self, raw_dir, subdir_name="RAW_POSITIVES"):
        self.raw_dir = Path(raw_dir)
        self.subdir_name = subdir_name
        self.target_dir = self.raw_dir / subdir_name
        self.image_paths = sorted(list(self.target_dir.glob("*.jpg")))
        
    def __len__(self):
        return len(self.image_paths)
    
    def parse_yolo_bbox(self, txt_path):
        if not txt_path.exists():
            return None
        try:
            with open(txt_path, 'r') as f:
                lines = f.readlines()
            if not lines:
                return None
            parts = lines[0].strip().split()
            if len(parts) == 5:
                return list(map(float, parts[1:]))
        except Exception:
            pass
        return None

    def get_adaptive_crop(self, image, bbox, bbox_area_ratio):
        img_w, img_h = image.size
        cx, cy, w, h = bbox
        
        if bbox_area_ratio < 0.01:
            padding_factor = 4.0
        elif bbox_area_ratio < 0.05:
            padding_factor = 2.0
        else:
            padding_factor = 1.2
            
        crop_size = max(w * img_w, h * img_h) * padding_factor
        crop_size = max(crop_size, 64)
        
        cx_px, cy_px = cx * img_w, cy * img_h
        
        x1 = max(0, int(cx_px - crop_size / 2))
        y1 = max(0, int(cy_px - crop_size / 2))
        x2 = min(img_w, int(cx_px + crop_size / 2))
        y2 = min(img_h, int(cy_px + crop_size / 2))
        
        return image.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        if self.subdir_name == "RAW_NEGATIVES":
            bbox = None
        else:
            txt_path = img_path.with_suffix(".txt")
            bbox = self.parse_yolo_bbox(txt_path)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224))
            
        if bbox is not None:
            w, h = bbox[2], bbox[3]
            bbox_area_ratio = w * h
            crop_img = self.get_adaptive_crop(image, bbox, bbox_area_ratio)
        else:
            bbox_area_ratio = 0.0
            crop_img = Image.new("RGB", (224, 224))
            
        full_tensor = IMAGE_TRANSFORM(image)
        crop_tensor = IMAGE_TRANSFORM(crop_img)
        
        return full_tensor, crop_tensor, torch.tensor(bbox_area_ratio, dtype=torch.float32), img_path.name

def generate_and_save_embeddings(dataset_input_dir, model, split_name="train", subdir_name="RAW_POSITIVES"):
    """Generates embeddings directly from Kaggle input directory without unzipping."""
    print(f"\n" + "="*50)
    print(f"GENERATING EMBEDDINGS FOR: {split_name.upper()} ({subdir_name})")
    print(f"Reading directly from: {dataset_input_dir}")
    print(f"="*50)
    
    # Створюємо датасет прямо з папки Kaggle Input (без розпакування)
    dataset = DroneEmbeddingDataset(dataset_input_dir, subdir_name=subdir_name)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    embeddings_dict = {}
    
    print("\nExtracting embeddings via DINOv2 model...")
    progress_bar = tqdm(dataloader, desc="DINOv2 Inference", total=len(dataloader))
    
    with torch.no_grad():
        for full_imgs, crop_imgs, area_ratios, filenames in progress_bar:
            full_imgs = full_imgs.to(DEVICE)
            crop_imgs = crop_imgs.to(DEVICE)
            area_ratios = area_ratios.to(DEVICE)
            
            emb_full = model(full_imgs) 
            emb_crop = model(crop_imgs) 
            
            emb_full = torch.nn.functional.normalize(emb_full, p=2, dim=1)
            emb_crop = torch.nn.functional.normalize(emb_crop, p=2, dim=1)
            
            w_full = torch.where(area_ratios == 0.0, torch.ones_like(area_ratios), area_ratios).unsqueeze(1)
            w_crop = torch.where(area_ratios == 0.0, torch.zeros_like(area_ratios), 1.0 - area_ratios).unsqueeze(1)
            
            emb_full_weighted = emb_full * w_full
            emb_crop_weighted = emb_crop * w_crop
            
            final_embeddings = torch.cat([emb_full_weighted, emb_crop_weighted], dim=1)
            final_embeddings = F.normalize(final_embeddings, p=2, dim=1)
            final_embeddings = final_embeddings.cpu().numpy()

            for name, emb in zip(filenames, final_embeddings):
                embeddings_dict[name] = emb
                
    suffix = "positives" if subdir_name == "RAW_POSITIVES" else "negatives"
    output_file_name = f"{split_name}_{suffix}_embeddings.pt"
    output_file_path = OUTPUT_DIR / output_file_name
    
    print(f"\nSaving embeddings dictionary to {output_file_path}...")
    torch.save(embeddings_dict, output_file_path)
    
    return output_file_path

In [6]:
if __name__ == "__main__":
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN not found in Kaggle Secrets.")

    api = HfApi()
    
    print("Loading DINOv2 ViT-B/14 model on GPU...")
    dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').to(DEVICE)
    dinov2_model.eval()
    
    TARGET_SUBDIR = "RAW_NEGATIVES"
    
    # Шлях до твого підключеного датасету (Kaggle Input)
    dataset_folder = Path("/kaggle/input/datasets/hannusiaknysh/train-drone-detection/extracted_dataset/train")

    
    if not dataset_folder.exists():
        raise FileNotFoundError(f"Could not find dataset directory at {dataset_folder}. Make sure it is attached to the notebook.")

    # Запускаємо процес генерації безпосередньо з папки
    emb_file_path = generate_and_save_embeddings(
        dataset_input_dir=dataset_folder, 
        model=dinov2_model,
        split_name="train",
        subdir_name=TARGET_SUBDIR
    )
    
    print(f"\nUploading resulting embeddings file '{emb_file_path.name}' to Hugging Face...")
    api.upload_file(
        path_or_fileobj=str(emb_file_path),
        path_in_repo=f"embeddings/{emb_file_path.name}",
        repo_id=REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN
    )
    
    print("\n" + "="*50)
    print("SUCCESS! EMBEDDINGS EXTRACTED AND UPLOADED TO HUGGING FACE!")
    print("="*50)

Loading DINOv2 ViT-B/14 model on GPU...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
Exception ignored in: <function _ConnectionBase.__del__ at 0x7f13f3e0f560>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 133, in __del__
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    _close(self._handle)
OSError: [Errno 9] Bad fi


GENERATING EMBEDDINGS FOR: TRAIN (RAW_NEGATIVES)
Reading directly from: /kaggle/input/datasets/hannusiaknysh/train-drone-detection/extracted_dataset/train

Extracting embeddings via DINOv2 model...


DINOv2 Inference:   0%|          | 0/93 [00:00<?, ?it/s]


Saving embeddings dictionary to /kaggle/working/embeddings/train_negatives_embeddings.pt...

Uploading resulting embeddings file 'train_negatives_embeddings.pt' to Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


SUCCESS! EMBEDDINGS EXTRACTED AND UPLOADED TO HUGGING FACE!
